## Import Requirements
#### if libraries are not available please install them locally using(pip install scikit-learn)

In [137]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import optuna
import joblib
import time
import re
import nltk


from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline 
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from optuna.integration.mlflow import MLflowCallback


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,f1_score

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4" 

import warnings
warnings.filterwarnings("ignore")

## Data cleaning (EDA) 

In [138]:
df=pd.read_csv("data.csv")

In [139]:
df

,Reviewer Name,Review Title,Place of Review,Up Votes,Down Votes,Month,Review text,Ratings
0,Kamal Suresh,Nice product,"Certified Buyer, Chirakkal",889.0,64.0,Feb 2021,"Nice product, good quality, but price is now r...",4
1,Flipkart Customer,Don't waste your money,"Certified Buyer, Hyderabad",109.0,6.0,Feb 2021,They didn't supplied Yonex Mavis 350. Outside ...,1
2,A. S. Raja Srinivasan,Did not meet expectations,"Certified Buyer, Dharmapuri",42.0,3.0,Apr 2021,Worst product. Damaged shuttlecocks packed in ...,1
3,Suresh Narayanasamy,Fair,"Certified Buyer, Chennai",25.0,1.0,NaN,"Quite O. K. , but nowadays the quality of the...",3
4,ASHIK P A,Over priced,NaN,147.0,24.0,Apr 2016,Over pricedJust â?¹620 ..from retailer.I didn'...,1
...,...,...,...,...,...,...,...,...
8513,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
8514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
8515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
8516,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [140]:
df.isnull().sum()/len(df)*100

Reviewer Name      0.117398
Review Title       0.117398
Place of Review    0.586992
Up Votes           0.117398
Down Votes         0.117398
Month              5.459028
Review text        0.093919
Ratings            0.000000
dtype: float64

In [141]:
df = df.drop(columns=['Reviewer Name','Place of Review','Month'])
df = df.dropna(subset=['Review text'])
initial_count = len(df)
df = df.drop_duplicates(subset=['Review text'], keep='first')
print(f"Removed {initial_count - len(df)} duplicate reviews.")
df['Review Title'] = df['Review Title'].fillna('')
df['Up Votes'] = df['Up Votes'].fillna(0)
df['Down Votes'] = df['Down Votes'].fillna(0)
df['full_review'] = df['Review Title'] + ' ' + df['Review text']
df = df.reset_index(drop=True)
df.isnull().sum()

Removed 3527 duplicate reviews.


Review Title    0
Up Votes        0
Down Votes      0
Review text     0
Ratings         0
full_review     0
dtype: int64

In [142]:
df.shape

(4983, 6)

In [143]:
df = df[df['Ratings'] != 3]
df['sentiment'] = df['Ratings'].apply(lambda x: 1 if x >= 4 else 0)
df = df.drop(columns=['Ratings'])
df = df.reset_index(drop=True)

In [144]:

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
negations = {'not', 'no', 'nor', 'never'}
stop_words = stop_words - negations
lemmatizer = WordNetLemmatizer()
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [
        lemmatizer.lemmatize(word)
        for word in text.split()
        if word not in stop_words and len(word) > 2]
    return ' '.join(tokens)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [145]:
df['clean_review'] = df['full_review'].apply(clean_text)
df[['full_review', 'clean_review']].sample(5)


,full_review,clean_review
2011,Fair SatisfactionREAD MORE,fair satisfactionread
2360,Delightful Nice 👌👌👌👌READ MORE,delightful nice read
430,Very Good Super 👍👍👍READ MORE,good super read
4585,Costly on flipkart You will get this product a...,costly flipkart get product around max retaile...
401,Terrific purchase Great product. Timely delive...,terrific purchase great product timely deliver...


## Train–Test Split

In [146]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

##  CV strategy

In [147]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


##  MODEL 1: Logistic Regression

In [148]:
def objective_logreg(trial):
    tfidf_max_features = trial.suggest_int("max_features", 5000, 20000)
    ngram_range = trial.suggest_categorical("ngram_range", [(1,1), (1,2)])
    C = trial.suggest_float("C", 0.1, 10.0, log=True)

    pipeline = ImbPipeline([
        ("tfidf", TfidfVectorizer(max_features=tfidf_max_features,ngram_range=ngram_range,sublinear_tf=True,
            min_df=3)),
        ("smote", SMOTE(random_state=42)), 
        ("clf", LogisticRegression(C=C, max_iter=1000))])

    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_macro")
    return scores.mean()

## Optuna-Run

In [149]:
study_logreg = optuna.create_study(direction="maximize")
study_logreg.optimize(objective_logreg, n_trials=30)

logreg_f1 = study_logreg.best_value
logreg_params = study_logreg.best_params


[I 2026-02-02 08:25:15,062] A new study created in memory with name: no-name-1ae6d005-f952-4178-a34d-72124a4addd6
[I 2026-02-02 08:25:15,796] Trial 0 finished with value: 0.8459317998787498 and parameters: {'max_features': 16145, 'ngram_range': (1, 2), 'C': 0.49727515530759264}. Best is trial 0 with value: 0.8459317998787498.
[I 2026-02-02 08:25:16,244] Trial 1 finished with value: 0.8436679155864084 and parameters: {'max_features': 7716, 'ngram_range': (1, 2), 'C': 0.3234786065641758}. Best is trial 0 with value: 0.8459317998787498.
[I 2026-02-02 08:25:16,554] Trial 2 finished with value: 0.8429650178310799 and parameters: {'max_features': 10302, 'ngram_range': (1, 1), 'C': 1.0308703500865772}. Best is trial 0 with value: 0.8459317998787498.
[I 2026-02-02 08:25:17,145] Trial 3 finished with value: 0.8460810832455211 and parameters: {'max_features': 19431, 'ngram_range': (1, 2), 'C': 6.947769584309274}. Best is trial 3 with value: 0.8460810832455211.
[I 2026-02-02 08:25:17,481] Trial 4

## MODEL 2: Linear SVM

In [150]:
def objective_svm(trial):
    tfidf_max_features = trial.suggest_int("max_features", 5000, 20000)
    ngram_range = trial.suggest_categorical("ngram_range", [(1,1), (1,2)])
    C = trial.suggest_float("C", 1e-3, 10, log=True)
    pipeline = ImbPipeline([
        ("tfidf", TfidfVectorizer(max_features=tfidf_max_features,ngram_range=ngram_range,sublinear_tf=True, min_df=3)),
        ("smote", SMOTE(random_state=42)), # Generates synthetic negative reviews
        ("clf", LinearSVC(C=C, max_iter=2000)) ])

    scores = cross_val_score(pipeline,X_train,y_train,cv=cv,scoring="f1_macro")

    return scores.mean()

In [151]:
study_svm = optuna.create_study(direction="maximize")
study_svm.optimize(objective_svm, n_trials=30)

svm_f1 = study_svm.best_value
svm_params = study_svm.best_params

[I 2026-02-02 08:25:53,079] A new study created in memory with name: no-name-7ea2a4e4-edbd-4ebf-92e2-975ce0e4e134
[I 2026-02-02 08:25:53,566] Trial 0 finished with value: 0.8543459374393543 and parameters: {'max_features': 11534, 'ngram_range': (1, 2), 'C': 0.08756844701321756}. Best is trial 0 with value: 0.8543459374393543.
[I 2026-02-02 08:25:54,023] Trial 1 finished with value: 0.8342699978477345 and parameters: {'max_features': 5834, 'ngram_range': (1, 2), 'C': 1.7691798132604661}. Best is trial 0 with value: 0.8543459374393543.
[I 2026-02-02 08:25:54,297] Trial 2 finished with value: 0.8451662252836663 and parameters: {'max_features': 18445, 'ngram_range': (1, 1), 'C': 0.05400104682370516}. Best is trial 0 with value: 0.8543459374393543.
[I 2026-02-02 08:25:54,699] Trial 3 finished with value: 0.8288804390823759 and parameters: {'max_features': 15248, 'ngram_range': (1, 2), 'C': 0.004497480987244084}. Best is trial 0 with value: 0.8543459374393543.
[I 2026-02-02 08:25:55,161] Tri

## MODEL 3: Naive Bayes

In [153]:

def objective_nb(trial):
    tfidf_max_features = trial.suggest_int("max_features", 5000, 20000)
    ngram_range = trial.suggest_categorical("ngram_range", [(1,1), (1,2)])
    alpha = trial.suggest_float("alpha", 1e-3, 1.0, log=True)

    pipeline = ImbPipeline([
        ("tfidf", TfidfVectorizer( max_features=tfidf_max_features,ngram_range=ngram_range,sublinear_tf=True,min_df=3)),
        ("smote", SMOTE(random_state=42)),
        ("scaler", MaxAbsScaler()), 
        ("clf", MultinomialNB(alpha=alpha))])

    scores = cross_val_score(pipeline,X_train,y_train,cv=cv,scoring="f1_macro")

    return scores.mean()

In [154]:
study_nb = optuna.create_study(direction="maximize")
study_nb.optimize(objective_nb, n_trials=30)

nb_f1 = study_nb.best_value
nb_params = study_nb.best_params

[I 2026-02-02 08:26:19,370] A new study created in memory with name: no-name-641c8696-baff-4265-a933-d6d30a77e35f
[I 2026-02-02 08:26:19,989] Trial 0 finished with value: 0.8366763525711052 and parameters: {'max_features': 6845, 'ngram_range': (1, 2), 'alpha': 0.0033858838984198135}. Best is trial 0 with value: 0.8366763525711052.
[I 2026-02-02 08:26:20,245] Trial 1 finished with value: 0.7917857609455039 and parameters: {'max_features': 11574, 'ngram_range': (1, 1), 'alpha': 0.3408407260020939}. Best is trial 0 with value: 0.8366763525711052.
[I 2026-02-02 08:26:20,503] Trial 2 finished with value: 0.7874424013720621 and parameters: {'max_features': 19041, 'ngram_range': (1, 1), 'alpha': 0.022509508289834328}. Best is trial 0 with value: 0.8366763525711052.
[I 2026-02-02 08:26:20,892] Trial 3 finished with value: 0.8349379768499461 and parameters: {'max_features': 13123, 'ngram_range': (1, 2), 'alpha': 0.103696896501864}. Best is trial 0 with value: 0.8366763525711052.
[I 2026-02-02 0

## Compare models (CV F1 only)

In [155]:

all_studies = {"Logistic Regression": study_logreg,"Linear SVM": study_svm,"Naive Bayes": study_nb}

results = {"Logistic Regression": study_logreg.best_value,"Linear SVM": study_svm.best_value,"Naive Bayes": study_nb.best_value}

In [156]:
results = {
    "Logistic Regression": logreg_f1,
    "Linear SVM": svm_f1,
    "Naive Bayes": nb_f1}

results


{'Logistic Regression': 0.8549255275653236,
 'Linear SVM': 0.857650856445115,
 'Naive Bayes': 0.8366763525711052}

## Train BEST model on full training data

In [157]:

best_model_name = max(results, key=results.get)
print(f" The winner based on your data is: {best_model_name}")


final_params = all_studies[best_model_name].best_params

if best_model_name == "Linear SVM":
    clf = LinearSVC(C=final_params["C"])
elif best_model_name == "Logistic Regression":
    clf = LogisticRegression(C=final_params["C"], max_iter=1000)
elif best_model_name == "Naive Bayes":
    clf = MultinomialNB(alpha=final_params["alpha"])

steps = [
    ("tfidf", TfidfVectorizer(
        max_features=final_params["max_features"],
        ngram_range=final_params["ngram_range"],
        sublinear_tf=True,
        min_df=3)),
    ("smote", SMOTE(random_state=42))
]
if best_model_name == "Naive Bayes":
    steps.append(("scaler", MaxAbsScaler()))

steps.append(("clf", clf))

best_pipeline = ImbPipeline(steps)
best_pipeline.fit(X_train, y_train)
print("Best pipeline trained successfully!")

 The winner based on your data is: Linear SVM
Best pipeline trained successfully!


## Best model setup for sentiment classification

In [110]:
# Linear SVM was your best performer
best_pipeline = ImbPipeline([
    ("tfidf", TfidfVectorizer(
        max_features=svm_params["max_features"],
        ngram_range=svm_params["ngram_range"],
        sublinear_tf=True,
        min_df=3)),
    ("smote", SMOTE(random_state=42)),
    ("clf", LinearSVC(C=svm_params["C"]))])

best_pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=12480, min_df=3,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('smote', SMOTE(random_state=42)),
                ('clf', LinearSVC(C=0.2569493393250539))])

## FINAL TEST EVALUATION

In [158]:

y_test_pred = best_pipeline.predict(X_test)

print(classification_report(y_test, y_test_pred))
print("Final Test F1:", f1_score(y_test, y_test_pred))


              precision    recall  f1-score   support

           0       0.76      0.88      0.82       179
           1       0.97      0.93      0.95       740

    accuracy                           0.92       919
   macro avg       0.87      0.91      0.88       919
weighted avg       0.93      0.92      0.92       919

Final Test F1: 0.9511355815554026


### Download Model 

In [159]:

joblib.dump(best_pipeline, "sentiment_model.pkl")


['sentiment_model.pkl']

In [160]:
results = {"Logistic Regression": logreg_f1,"Linear SVM": svm_f1,"Naive Bayes": nb_f1}

best_model_name = max(results, key=results.get)
best_cv_f1 = results[best_model_name]

print("Best Model (based on CV F1):", best_model_name)
print("Best Cross-Validated F1:", round(best_cv_f1, 4))


Best Model (based on CV F1): Linear SVM
Best Cross-Validated F1: 0.8577


In [161]:
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1_macro = f1_score(y_test, y_test_pred, average="macro")
test_f1_positive = f1_score(y_test, y_test_pred) 

print("\nFinal Test Evaluation")
print("---------------------")
print("Selected Model:", best_model_name)
print("Test Accuracy:", round(test_accuracy, 4))
print("Test Macro F1:", round(test_f1_macro, 4))
print("Test Positive-class F1:", round(test_f1_positive, 4))



Final Test Evaluation
---------------------
Selected Model: Linear SVM
Test Accuracy: 0.9227
Test Macro F1: 0.8834
Test Positive-class F1: 0.9511


In [162]:
test_results = X_test.to_frame(name="review")
test_results["actual_sentiment"] = y_test.values
test_results["predicted_sentiment"] = y_test_pred

negative_reviews = test_results[(test_results["actual_sentiment"] == 0) &(test_results["predicted_sentiment"] == 0)]

print("Negative reviews analyzed :", len(negative_reviews))


Negative reviews analyzed : 157


### Top customer pain points from negative reviews

In [164]:
tfidf = TfidfVectorizer(
    max_features=15,
    stop_words="english",
    ngram_range=(1,2))

X_neg = tfidf.fit_transform(negative_reviews["review"])
pain_points = tfidf.get_feature_names_out()

print("Top customer pain points from negative reviews:")
for p in pain_points:
    print("-", p)


Top customer pain points from negative reviews:
- bad
- box
- buy
- dont
- experience
- good
- money
- poor
- product
- purchase
- quality
- qualityread
- shuttle
- waste
- worst
